## pipeline_module.py

In [5]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from transformers import pipeline as hf_pipeline

import torch
torch.set_num_threads(1)
# ----------------------------------------------------------------------
# 1. NLP 모듈: 텍스트 감성 점수 추출
# ----------------------------------------------------------------------
class NLPExtractor:
    """Hugging Face 기반의 KR-FINBERT를 사용하여 감성 점수를 추출하는 모듈."""

    def __init__(self, model_name="snunlp/KR-FINBERT-SC"):
        print(f"✅ NLP 모듈 초기화: {model_name} 로드 중...")
        # 'text-classification' 파이프라인 로드
        self.classifier = hf_pipeline(
            "text-classification",
            model=model_name,
            truncation=True,
            max_length=128,
            device=-1
        )

    def extract_sentiment(self, texts: list[str]) -> pd.DataFrame:
        """
        텍스트 리스트를 입력받아 감성 점수를 반환합니다.

        반환: pd.DataFrame (label, score)
        """
        if not texts:
            return pd.DataFrame(columns=['label', 'score'])

        # FinBERT 추론 실행
        results = self.classifier(texts, truncation=True)

        scores = [{
            'sentiment_label': res['label'],
            'sentiment_score': res['score'] * (1 if res['label'] == '긍정' else -1)  # 긍정: +, 부정: -
        } for res in results]

        return pd.DataFrame(scores)


# ----------------------------------------------------------------------
# 2. GCN 모듈: Node Embedding 추출
# ----------------------------------------------------------------------
class GCNFeatureExtractor(torch.nn.Module):
    """
    GCN 모델 구조 정의. 이 모델의 최종 출력 (32차원 벡터)이
    'Node Embedding'으로 XGBoost/LightGBM으로 전달됩니다.
    """

    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        print(f"✅ GCN 초기화: 입력={in_channels}, 임베딩 차원={out_channels}")

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x


class GCNPipeline:
    """GCN 모델을 통해 초기 피처와 관계로부터 Node Embedding을 추출하는 파이프라인."""

    def __init__(self, in_dim: int, out_dim: int = 32, model_path: str = None):
        self.model = GCNFeatureExtractor(in_channels=in_dim, hidden_channels=64, out_channels=out_dim)

        # 학습된 가중치 로드 (실제 운영 환경)
        if model_path:
            try:
                self.model.load_state_dict(torch.load(model_path))
                self.model.eval()
            except FileNotFoundError:
                print(f"경고: 학습된 GCN 가중치 파일({model_path})을 찾을 수 없습니다. 무작위 가중치로 실행됩니다.")

    def _create_graph_data(self, features_df: pd.DataFrame, relations_df: pd.DataFrame) -> Data:
        """DataFrame을 PyG Data 객체로 변환합니다."""

        # features_df의 인덱스를 노드 ID로 사용한다고 가정합니다.
        # 1. Node Features (초기 피처) 생성
        features_tensor = torch.tensor(features_df.values, dtype=torch.float)

        # 2. Edge Index 생성 (종목 관계)
        # DataFrame 인덱스(0부터 시작)를 사용해야 합니다.
        node_map = {ticker: i for i, ticker in enumerate(features_df.index)}

        source_nodes = relations_df['from_ticker'].map(node_map).values
        target_nodes = relations_df['to_ticker'].map(node_map).values

        edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)

        data = Data(x=features_tensor, edge_index=edge_index)
        print(f"✅ 그래프 데이터 생성 완료. 노드 수: {data.num_nodes}, 엣지 수: {data.num_edges}")
        return data

    def extract_embedding(self, initial_features: pd.DataFrame, relations_df: pd.DataFrame) -> pd.DataFrame:
        """
        GCN 추론을 통해 Node Embedding을 추출하고 DataFrame 형태로 반환합니다.
        """
        graph_data = self._create_graph_data(initial_features, relations_df)

        self.model.eval()  # 추론 모드
        with torch.no_grad():
            # [노드 수, out_dim] 형태의 Node Embedding 텐서
            node_embeddings = self.model(graph_data.x, graph_data.edge_index).numpy()

        # 결과를 DataFrame으로 변환하여 반환
        embedding_df = pd.DataFrame(
            node_embeddings,
            index=initial_features.index,
            columns=[f'gcn_emb_{i}' for i in range(self.model.conv2.out_channels)]
        )
        print(f"✅ GCN Node Embedding 추출 완료. 형태: {embedding_df.shape}")
        return embedding_df


# ----------------------------------------------------------------------
# 3. XGBoost 모듈: 최종 학습 및 예측
# ----------------------------------------------------------------------
class XGBoostModel:
    """최종 피처를 입력받아 XGBoost를 학습/추론하는 모듈."""

    def __init__(self, params: dict):
        self.model = xgb.XGBClassifier(**params, random_state=42)

    def train_and_evaluate(self, X: pd.DataFrame, y: pd.Series, test_size=0.2):
        """데이터를 분할하고 모델을 학습 및 평가합니다."""
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state=42,
            stratify=y
        )

        print(f"✅ XGBoost 학습 데이터 크기: {len(X_train)}")
        self.model.fit(X_train, y_train)

        y_pred = self.model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        print(f"--- XGBoost 결과 ---")
        print(f"테스트 데이터 정확도 (Accuracy): {accuracy:.4f}")
        print(f"----------------------")
        return accuracy

In [6]:
!where pip

c:\Users\user\project\ai\venv\Scripts\pip.exe
C:\Users\user\anaconda3\Scripts\pip.exe
C:\Users\user\AppData\Local\Programs\Python\Python313\Scripts\pip.exe


In [8]:
import requests
from bs4 import BeautifulSoup
from elasticsearch import Elasticsearch
from transformers import pipeline as hf_pipeline
import time

# --------------------------
# 1. 텍스트 500자 청크
# --------------------------
def split_text_into_chunks(text, max_len=500):
    return [text[i:i+max_len] for i in range(0, len(text), max_len)]


# --------------------------
# 2. 네이버 뉴스 본문 크롤링
# --------------------------
def fetch_naver_news_text(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")
    article = soup.select_one("#dic_area")
    return article.get_text(separator="\n").strip() if article else ""


# --------------------------
# 3. 네이버 뉴스 API (news.naver.com만)
# --------------------------
def fetch_stock_news_from_naver_api(query="증시", display=20):
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id": "UnXIIDYID51p2ZYlnY24",
        "X-Naver-Client-Secret": "9Z1WEzpQR3"
    }
    params = {"query": query, "display": display, "sort": "date"}

    res = requests.get(url, headers=headers, params=params)
    items = res.json().get("items", [])
    urls = []

    for item in items:
        link = item.get("link", "")
        if "news.naver.com" in link:
            urls.append(link)

    return urls


# --------------------------
# 4. FinBERT 감성 분석 모델
# --------------------------
sentiment_model = hf_pipeline(
    "text-classification",
    model="snunlp/KR-FINBERT-SC",
    truncation=True,
    max_length=128,
    device=-1
)


# --------------------------
# 5. ES 클라이언트 연결
# --------------------------
es = Elasticsearch(
    hosts=["http://localhost:9200"],  # ES 주소
)


# --------------------------
# 6. 뉴스 → ES 저장 ETL
# --------------------------
def run_news_etl():

    print("📌 뉴스 수집 시작")

    urls = fetch_stock_news_from_naver_api()
    print(f"👉 수집된 네이버 뉴스 URL {len(urls)}개")

    for url in urls:
        print(f"▶ 처리 중: {url}")

        article_text = fetch_naver_news_text(url)
        if not article_text:
            continue

        chunks = split_text_into_chunks(article_text, max_len=500)

        # 감성 분석
        sentiments = sentiment_model(chunks)

        # 감성 점수 정제
        sentiment_scores = [
            (s["score"] if s["label"] == "긍정" else -s["score"])
            for s in sentiments
        ]

        doc = {
            "url": url,
            "text": article_text,
            "chunks": chunks,
            "sentiments": sentiment_scores,
            "timestamp": time.time()
        }

        es.index(index="news_articles", document=doc)

    print("📌 ETL 완료 (ES 저장 완료)")


Device set to use cpu


In [12]:
import sys
sys.path.append(r"C:\Users\user\project\MyEggBasket-AI")

from elasticsearch import Elasticsearch
from pipeline_module import NLPExtractor, GCNPipeline, XGBoostModel
import pandas as pd
import numpy as np



# --------------------------
# ES에서 뉴스 불러오기
# --------------------------
def load_news_from_es(limit=100):
    es = Elasticsearch("http://localhost:9200")

    res = es.search(
        index="news_articles",
        size=limit,
        query={"match_all": {}}
    )

    news_list = [hit["_source"] for hit in res["hits"]["hits"]]
    return news_list



def run_pipeline():
    print("=============================================")
    print("🚀 금융 AI 파이프라인 시작")
    print("=============================================")

    # --------------------------
    # A. ES에서 뉴스 읽기
    # --------------------------
    news_data = load_news_from_es(limit=50)

    if len(news_data) == 0:
        print("❌ ES에 뉴스 데이터가 없음. ETL 먼저 실행하세요.")
        return

    print(f"👉 ES에서 뉴스 {len(news_data)}개 로드")

    # NLP 피처로 사용할 감성 점수만 가져오기
    sentiment_features = []
    for n in news_data:
        sentiment_features.extend(n["sentiments"])

    sentiment_df = pd.DataFrame({"sentiment": sentiment_features[:100]})

    # --------------------------
    # B. 더미 GCN 입력 데이터 생성
    # --------------------------
    TICKER_COUNT = 100
    TIME_STEPS = 100
    FEATURE_COUNT = 10

    ticker_list = [f'TKR{i:03d}' for i in range(TICKER_COUNT)]
    initial_features = pd.DataFrame(
        np.random.rand(TICKER_COUNT, FEATURE_COUNT),
        index=ticker_list,
        columns=[f'initial_feature_{i}' for i in range(FEATURE_COUNT)]
    )

    relations = pd.DataFrame({
        'from_ticker': np.random.choice(ticker_list, 1000),
        'to_ticker': np.random.choice(ticker_list, 1000),
        'weight': np.random.rand(1000)
    })

    # --------------------------
    # C. GCN 실행
    # --------------------------
    gcn = GCNPipeline(in_dim=FEATURE_COUNT, out_dim=32)
    gcn_df = gcn.extract_embedding(initial_features, relations)

    # --------------------------
    # D. XGBoost 학습
    # --------------------------
    N_SAMPLES = TICKER_COUNT * TIME_STEPS
    FINAL_FEATURE_COUNT = 120

    X_final = pd.DataFrame(
        np.random.randn(N_SAMPLES, FINAL_FEATURE_COUNT),
        columns=[f'feature_{i}' for i in range(FINAL_FEATURE_COUNT)]
    )
    y_final = pd.Series(np.random.randint(0, 2, N_SAMPLES))

    xgb = XGBoostModel(params={
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "eta": 0.1,
        "max_depth": 6
    })

    xgb.train_and_evaluate(X_final, y_final)


ModuleNotFoundError: No module named 'pipeline_module'

## main.py (주윤)

In [8]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def split_text_into_chunks(text, max_len=500):
    chunks = []
    for i in range(0, len(text), max_len):
        chunks.append(text[i:i+max_len])
    return chunks



def fetch_naver_news_text(url):
    """
    네이버 뉴스 본문 크롤링
    """
    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    article = soup.select_one("#dic_area")
    if article:
        return article.get_text(separator="\n").strip()
    return ""


def fetch_stock_news_from_naver_api(query="주식", display=20):
    """
    네이버 뉴스 API에서 기사 URL 가져오기
    """
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id": "UnXIIDYID51p2ZYlnY24",
        "X-Naver-Client-Secret": "9Z1WEzpQR3"
    }
    params = {"query": query, "display": display, "sort": "date"}

    res = requests.get(url, headers=headers, params=params)

    if res.status_code != 200:
        print("❌ 네이버 API 요청 실패:", res.text)
        return []

    items = res.json().get("items", [])
    news_urls = []

    for item in items:
        link = item.get("link", "")

        # 🔥 핵심: 네이버 뉴스 페이지만 가져오기
        if "news.naver.com" in link:
            news_urls.append(link)

    return news_urls




def run_pipeline():
    """
    파이프라인의 각 모듈을 순차적으로 실행하고, 데이터를 연결하여 최종 XGBoost에 전달합니다.
    """
    print("=============================================")
    print("         🚀 금융 AI 파이프라인 시작         ")
    print("=============================================")

    # ----------------------------------------------------------
    # A. 실제 뉴스 수집 + NLP용 chunk 생성
    # ----------------------------------------------------------
    print("\n[뉴스 수집 중...]")

    news_urls = fetch_stock_news_from_naver_api(query="증시", display=10)

    print(">>> API에서 받은 URL 목록:")
    for i, u in enumerate(news_urls):
        print(f"{i+1}. {u}")

    

    all_chunks = []
    for url in news_urls:
        article = fetch_naver_news_text(url)
        if article:
            chunks = split_text_into_chunks(article, max_len=500)
            all_chunks.extend(chunks)

    print(f"👉 전체 뉴스 chunk 개수: {len(all_chunks)}")
    if len(all_chunks) > 0:
        print("\n[예시 chunk 1]")
        print(all_chunks[0][:200], "...")



    # ----------------------------------------------------------------------
    # A. 더미 데이터 생성 (실제 DB 로드 대체)
    # ----------------------------------------------------------------------
    TICKER_COUNT = 100
    TIME_STEPS = 100
    FEATURE_COUNT = 10

    # GCN 입력 데이터 (초기 피처 & 관계)
    ticker_list = [f'TKR{i:03d}' for i in range(TICKER_COUNT)]

    initial_features = pd.DataFrame(
        np.random.rand(TICKER_COUNT, FEATURE_COUNT),
        index=ticker_list,
        columns=[f'initial_feature_{i}' for i in range(FEATURE_COUNT)]
    )

    relations = pd.DataFrame({
        'from_ticker': np.random.choice(ticker_list, 1000),
        'to_ticker': np.random.choice(ticker_list, 1000),
        'weight': np.random.rand(1000)
    })


    # ----------------------------------------------------------------------
    #  파이프라인 실행
    # ----------------------------------------------------------------------

    # 1. NLP 모듈 실행
    nlp_module = NLPExtractor()
    sentiment_results = nlp_module.extract_sentiment(all_chunks)
    print("\n[단계 1. NLP 결과 (샘플)]")
    print(sentiment_results[:3])

    # 2. GCN 모듈 실행
    # GCN 입력 차원은 initial_features의 컬럼 수
    gcn_module = GCNPipeline(in_dim=FEATURE_COUNT, out_dim=32)
    node_embeddings_df = gcn_module.extract_embedding(
        initial_features=initial_features,
        relations_df=relations
    )
    print("\n[단계 2. GCN Embedding 결과]")
    print(node_embeddings_df.head(3))

    # GCN Embedding 결과는 Feature Store에 저장될 최종 피처에 통합되어야 함.


    # XGBoost 학습 데이터 (Feature Store 시뮬레이션 - N_SAMPLES = 100 * 100 = 10000)
    N_SAMPLES = TICKER_COUNT * TIME_STEPS
    FINAL_FEATURE_COUNT = 120
    X_final = pd.DataFrame(
        np.random.randn(N_SAMPLES, FINAL_FEATURE_COUNT),
        columns=[f'feature_{i}' for i in range(FINAL_FEATURE_COUNT)]
    )
    y_final = pd.Series(np.random.randint(0, 2, N_SAMPLES), name='target')



    # (여기서는 X_final에 이미 통합되었다고 가정하고 XGBoost로 넘어갑니다.)

    # 3. XGBoost 모듈 실행 (더미 데이터로 학습 및 평가)
    xgb_module = XGBoostModel(params={
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'eta': 0.1,
        'max_depth': 6,
        'use_label_encoder': False
    })
    
    print("\n[단계 3. XGBoost 실행]")
    xgb_module.train_and_evaluate(X=X_final, y=y_final)

    print("=============================================")
    print("         ✅ 파이프라인 전체 실행 완료         ")
    print("=============================================")


if __name__ == '__main__':
    run_pipeline()

         🚀 금융 AI 파이프라인 시작         

[뉴스 수집 중...]
>>> API에서 받은 URL 목록:
1. https://n.news.naver.com/mnews/article/003/0013617095?sid=104
👉 전체 뉴스 chunk 개수: 3

[예시 chunk 1]
주요 물가·소비·주택지표 공개
추수감사절 단축 거래 속 주요 IT·유통 실적 이어져








[서울=뉴시스] 23일 야후파이낸스에 따르면 추수감사절 연휴로 단축 거래가 이뤄지는 이번 주 뉴욕증시는 주요 경제지표와 기업 실적이 대거 쏟아지며 바쁜 한 주를 보낼 전망이다. 사진은 서울 중구 하나은행 딜링룸에서 딜러들이 업무를 보고 있다.  2025.11. ...
✅ NLP 모듈 초기화: snunlp/KR-FINBERT-SC 로드 중...


Device set to use cpu



[단계 1. NLP 결과 (샘플)]
  sentiment_label  sentiment_score
0         neutral        -0.999921
1         neutral        -0.974794
2         neutral        -0.999918
✅ GCN 초기화: 입력=10, 임베딩 차원=32
✅ 그래프 데이터 생성 완료. 노드 수: 100, 엣지 수: 1000
✅ GCN Node Embedding 추출 완료. 형태: (100, 32)

[단계 2. GCN Embedding 결과]
        gcn_emb_0  gcn_emb_1  gcn_emb_2  gcn_emb_3  gcn_emb_4  gcn_emb_5  \
TKR000   0.064474  -0.339193   0.292688  -0.051793   0.344420   0.270095   
TKR001   0.074038  -0.398940   0.332436  -0.062005   0.380564   0.296768   
TKR002   0.080042  -0.356589   0.300910  -0.049435   0.343656   0.277346   

        gcn_emb_6  gcn_emb_7  gcn_emb_8  gcn_emb_9  ...  gcn_emb_22  \
TKR000  -0.002569   0.102245  -0.427700  -0.035631  ...   -0.200646   
TKR001   0.005369   0.101230  -0.494382  -0.052981  ...   -0.247706   
TKR002   0.000951   0.093109  -0.447012  -0.029591  ...   -0.218351   

        gcn_emb_23  gcn_emb_24  gcn_emb_25  gcn_emb_26  gcn_emb_27  \
TKR000    0.313974   -0.124160   -0.266636  

c:\Users\user\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [10:22:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- XGBoost 결과 ---
테스트 데이터 정확도 (Accuracy): 0.5150
----------------------
         ✅ 파이프라인 전체 실행 완료         
